# Fine-tune on GPU

One-click training on **Colab**, **Vertex AI Workbench**, or any NVIDIA GPU environment.

| GPU | VRAM | Max Model | Batch | Seq |
|-----|------|-----------|-------|-----|
| T4 | 16 GB | 7B-4bit | 1 | 2048 |
| L4 | 24 GB | 14B-4bit | 1 | 2048 |
| A100 40GB | 40 GB | 32B-4bit | 1 | 4096 |
| A100 80GB | 80 GB | 32B-4bit | 4 | 4096 |

In [ ]:
#@title Setup { display-mode: "form" }
#@markdown Clone repo and install Unsloth + CUDA dependencies.
#@markdown **First run will install packages and may restart the kernel — just re-run all cells.**

import os, subprocess, sys

# — Clone —
REPO = "https://github.com/piotrjanik/ai-tuner.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
if not os.path.exists("ai-tuner"):
    !git clone --depth 1 --branch {BRANCH} {REPO}
%cd ai-tuner

# — Ensure PyTorch has CUDA support —
_need_restart = False
try:
    import torch
    if not torch.cuda.is_available():
        raise ImportError("CPU-only torch")
except (ImportError, AssertionError):
    print("Installing PyTorch with CUDA support...")
    !pip install -q torch --index-url https://download.pytorch.org/whl/cu124
    _need_restart = True

# — Install uv for faster dependency resolution —
!pip install -q --upgrade uv

# — Install Unsloth (Colab-safe: pin numpy & pillow to avoid conflicts) —
try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil = f"pillow=={PIL.__version__}"
except Exception:
    _numpy = "numpy"
    _pil = "pillow"

!uv pip install -q --system --upgrade {_numpy} {_pil} bitsandbytes xformers unsloth
!uv pip install -q --system transformers==4.56.2
!uv pip install -q --system --no-deps trl==0.22.2

# — Install remaining project deps (peft, accelerate, typer, etc.) —
!uv pip install -q --system peft>=0.13.0 accelerate>=1.0.0 "typer[all]>=0.12.0" pyyaml tqdm "huggingface_hub[cli]" datasets

# — Flash Attention (optional, speeds up training on A100/H100) —
!timeout 60 pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "⚠ flash-attn not available — training will use eager attention instead"

# — Restart kernel if torch was reinstalled —
if _need_restart:
    print("
⚠ PyTorch was reinstalled with CUDA. Restarting kernel — please re-run all cells.")
    os.kill(os.getpid(), 9)

print("✓ Setup complete")

In [ ]:
# — GPU check —
import torch
assert torch.cuda.is_available(), "No CUDA GPU detected. Check that NVIDIA drivers are installed (run `nvidia-smi`)."
props = torch.cuda.get_device_properties(0)
print(f"✓ GPU: {props.name} ({props.total_memory / 1e9:.0f} GB)")
print(f"✓ Flash Attention 2: {'yes' if props.major >= 8 else 'no (eager fallback)'}")

In [ ]:
#@title Prepare data & train { display-mode: "form" }
#@markdown Prepares training data from config.yaml sources, then runs QLoRA training.
#@markdown Auto-tune picks batch size, seq length, and grad checkpoint based on your GPU.

!python src/data/prepare_data.py --config config.yaml
!python src/train/train_cuda.py --config config.yaml

In [ ]:
#@title Prepare data & train { display-mode: "form" }
#@markdown Prepares training data from config.yaml sources, then runs QLoRA training.
#@markdown Auto-tune picks batch size, seq length, and grad checkpoint based on your GPU.

!python src/cli.py data
!python src/cli.py train --cuda

In [ ]:
# (Optional) Push adapters to HuggingFace Hub
# !python src/cli.py push